# 6. Figures

## Import Packages

In [ ]:
import sys

sys.path.append("../../")
sys.path.append('../../src')

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import random
import matplotlib.pyplot as plt

from m3util.ml.rand import set_seeds
from m3util.viz.style import set_style
from m3util.viz.printing import printer
from belearn.viz.viz import Viz
from belearn.dataset.dataset import BE_Dataset
from belearn.functions.sho import SHO_nn

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

from m3util.viz.layout import inset_connector, add_box
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front

from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

printing = printer(basepath = './Figures/')


set_style("printing")
set_seeds(seed=42)

%matplotlib inline

In [ ]:
def SHO_fit_func_nn(params,
                    wvec_freq,
                    device='cpu'):
    """_summary_

    Returns:
        _type_: _description_
    """

    Amp = params[:, 0].type(torch.complex128)
    w_0 = params[:, 1].type(torch.complex128)
    Q = params[:, 2].type(torch.complex128)
    phi = params[:, 3].type(torch.complex128)
    wvec_freq = torch.tensor(wvec_freq)

    Amp = torch.unsqueeze(Amp, 1)
    w_0 = torch.unsqueeze(w_0, 1)
    phi = torch.unsqueeze(phi, 1)
    Q = torch.unsqueeze(Q, 1)

    wvec_freq = wvec_freq.to(device)

    numer = Amp * torch.exp((1.j) * phi) * torch.square(w_0)
    den_1 = torch.square(wvec_freq)
    den_2 = (1.j) * wvec_freq.to(device) * w_0 / Q
    den_3 = torch.square(w_0)

    den = den_1 - den_2 - den_3

    func = numer / den

    return func

## Loads Data

In [ ]:
# Specify the filename and the path to save the file
filename = "data_raw.h5"
save_path = "./Data"

data_path = save_path + "/" + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# print the contents of the file
dataset.print_be_tree()

In [ ]:
dataset.SHO_Scaler()

## Figure 1

In [ ]:
# instantiates the visualization object
BE_viz = Viz(dataset, printing, verbose=True)

In [ ]:
fig = BE_viz.raw_be(dataset, filename="Figure_2_raw_be_experiment")

In [ ]:
axes = fig.axes
axes

In [ ]:
def copy_axes_properties(source_ax, target_ax, secondary_ax=None, type_ax=None):
    """Copy properties and data from source_ax to target_ax."""
    target_ax.set_xlabel(source_ax.get_xlabel())
    target_ax.set_ylabel(source_ax.get_ylabel())
    target_ax.set_xlim(source_ax.get_xlim())
    target_ax.set_ylim(source_ax.get_ylim())
        
    for line in source_ax.get_lines():
        target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color())
        
    if secondary_ax != None:
        if type_ax == 'twin':
            ax_twin = target_ax.twinx()
            for line in secondary_ax.get_lines():
                ax_twin.plot(line.get_xdata(), line.get_ydata(), color=line.get_color())
        elif type_ax == 'same':
            for line in secondary_ax.get_lines():
                target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color())
                
                max_x_lim = max(source_ax.get_xlim()[1], secondary_ax.get_xlim()[1])
                min_x_lim = min(source_ax.get_xlim()[0], secondary_ax.get_xlim()[0])
                max_y_lim = max(source_ax.get_ylim()[1], secondary_ax.get_ylim()[1])
                min_y_lim = min(source_ax.get_ylim()[0], secondary_ax.get_ylim()[0])
                target_ax.set_xlim((min_x_lim, max_x_lim))
                target_ax.set_ylim((min_y_lim, max_y_lim))
        else:
            ax_new = target_ax.inset_axes([0.5, 0.65, 0.48, 0.33])
            x_start = 120
            x_end = 140
            ax_new.plot(dataset.hysteresis_waveform)
            ax_new.set_xlim(x_start, x_end)
            ax_new.set_ylim(0, -15)

            # drows the inset connector
            inset_connector(
                source_ax,
                target_ax,
                ax_new,
                [(x_start, 0), (x_end, 0)],
                [(x_start, 0), (x_end, 0)],
                color="k",
                linestyle="--",
                linewidth=0.5,
            )

            # adds a box on the figure
            add_box(
                target_ax,
                (x_start, 0, x_end, -15),
                edgecolor="k",
                linestyle="--",
                facecolor="none",
                linewidth=0.5,
                zorder=10,
            )
            ax_new.set_xlabel("Voltage Steps")
            ax_new.set_ylabel("Voltage (V)")

# Create a figure
fig = plt.figure(figsize=(14, 12))

# Define the GridSpec layout
gs = GridSpec(4, 4, figure=fig)

order = [['AFM'], [1], [0], [4, 'same', 6], [4, 'twin', 5], [2, 'inset', 3], ['hysteresis_loop']]

# List of axes indices in GridSpec for each subplot
subplot_specs = [(0, 2, 0, 2), # a (3D AFM)
                 (1, 2, 3, 4), # b (resonance frequency)
                 (1, 2, 2, 3), # c (waveform)
                 (0, 1, 2, 3), # d (real/imag)
                 (0, 1, 3, 4), # e (amp/phase)
                 (2, 4, 0, 2), # f (triangular waveform)
                 (2, 4, 2, 4)] # g (hysteresis)

# Create and set up subplots
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    ax = fig.add_subplot(gs[r_start:r_end, c_start:c_end])
    if i < len(axes):
        idx = order[i]
        if idx[0] == 'AFM':
            image_path = "/home/jca92/Rapid-Fitting-BEPFM-NN/notebooks/assets/AFM_field.png"
            img = mpimg.imread(image_path)
            ax.imshow(img)
            ax.axis('off')  # Turn off axis for image subplot
        elif idx[0] == 'hysteresis_loop':
            raw_hysteresis_loop, voltage = dataset.get_hysteresis(
                                loop_interpolated=True, plotting_values=True)
            row = random.randint(0, 59)
            col = random.randint(0, 59)
            cycle = random.randint(0, 3)
            ax.plot(voltage.squeeze()*-1,
                       raw_hysteresis_loop[row, col, cycle, :].squeeze())
            ax.set_xlabel("Amplitude (Arb. U.)")
            ax.set_ylabel("Voltage (V)")
        elif len(idx) == 1:
            copy_axes_properties(axes[idx[0]], ax)
        else:
            copy_axes_properties(axes[idx[0]], ax, axes[idx[2]], idx[1])
            

# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()

**Figure 1**: Schematic diagram of band-excitation piezoresponse force microscopy switching spectroscopy (BE-SS) **a** Artistic render of an AFM tip applying an electric field to the surface. **b** Band
of excited frequencies excited. The dashed line shows the cantilever resonance frequency. **c** Bandexcitation waveform used to excite the cantilever in time domain **d** Fast Fourier transform of a
single-cantilever resonance during band-excitation piezoresponse force microscopy – shown as real
and imaginary components. **e** Magnitude spectrum showing the amplitude and the phase of cantilever
resonance. **f** Bipolar-triangular waveform used to switch the ferroelectric. The inset shows where the
band-excitation waveform was applied in both the voltage-on and voltage-off states. **g** Example of a
typical piezoelectric hysteresis loop obtained during BE-SS.

## Figure 3

In [ ]:
set_seeds(seed=42)

postprocessor = ComplexPostProcessor(dataset)


model_ = Multiscale1DFitter(SHO_fit_func_nn, # function 
                            dataset.frequency_bin, # x data
                            2, # input channels
                            4, # output channels
                            dataset.SHO_scaler, 
                            postprocessor)




# instantiate the model
model = Model(model_, dataset, training=False, model_basename="SHO_Fitter_original_data")

model.load(
    "./Trained Models/SHO Fitter/2024-09-23_14-36-21_nn_benchmarks_noise/SHO_Fitter_model_optimizer_Adam_epoch_4_train_loss_0.040321211942850994.pth"
)

#model.load_state_dict(torch.load("./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_599_train_loss_0.005903387442231178.pth"))

X_data, Y_data = dataset.NN_data()

# you can view the test and training dataset by replacing X_data with X_test or X_train
pred_data, scaled_param, parm = model.predict(X_data)

fig = BE_viz.SHO_switching_maps(parm, filename="Figure_15_NN_Switching_Maps")

In [ ]:
axes = fig.axes

In [ ]:
true_state = {
    "fitter": "LSQF",
    "raw_format": "complex",
    "resampled": True,
    "scaled": True,
    "output_shape": "index",
    "measurement_state": "all",
}


fig = BE_viz.violin_plot_comparison_SHO(true_state, model, X_data, filename="Figure_16_Violin") 

In [ ]:
axes.extend(fig.axes)

In [ ]:
# sets the phase shift of the dataset
dataset.NN_phase_shift = np.pi/2
dataset.LSQF_phase_shift = np.pi/2
dataset.measurement_state = "all"

# sets the true state which to compare the results.
true_state = {
    "fitter": "LSQF",
    "raw_format": "complex",
    "resampled": True,
    "scaled": True,
    "output_shape": "index",
    "measurement_state": "all",
}

# sets the state of the output data
out_state = {"scaled": True, "raw_format": "magnitude spectrum"}

# sets the number of examples to get
n = 1

LSQF = BE_viz.get_best_median_worst(
    true_state,
    prediction={"fitter": "LSQF"},
    out_state=out_state,
    SHO_results=True,
    n=n,
)

NN = BE_viz.get_best_median_worst(
    true_state, prediction=model, out_state=out_state, SHO_results=True, n=n
)

data = (LSQF, NN)
names = ["LSQF", "NN"]

fig = BE_viz.SHO_Fit_comparison(
    data,
    names,
    model_comparison=[model, {"fitter": "LSQF"}],
    out_state=out_state,
    filename="Figure_14_LSQF_NN_bmw_comparison",
    # display_results = None
)

In [ ]:
axes.extend(fig.axes)

In [ ]:
axes=fig.axes

In [ ]:
axes

In [ ]:
import pandas as pd
import seaborn as sns
import itertools

def copy_axes_properties(source_ax, target_ax, secondary_ax=None, type_ax=None):
    """Copy properties and data from source_ax to target_ax."""
    # Copy basic properties
    target_ax.set_xlabel(source_ax.get_xlabel())
    target_ax.set_ylabel(source_ax.get_ylabel())
    target_ax.set_xlim(source_ax.get_xlim())
    target_ax.set_ylim(source_ax.get_ylim())
    target_ax.set_title(source_ax.get_title())

    # Copy lines from the primary axis
    for line in source_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Handle twin axes if present
    if secondary_ax:
        ax_twin = target_ax.twinx()
        ax_twin.set_ylim(secondary_ax.get_ylim())
        ax_twin.set_ylabel(secondary_ax.get_ylabel())

        for line in secondary_ax.get_lines():
            label = line.get_label() if line.get_label() != '_nolegend_' else None
            # Copying line properties for the twin axis
            ax_twin.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                         linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Copy legends
    if source_ax.get_legend():
        target_ax.legend(fontsize='large')

    if secondary_ax and secondary_ax.get_legend():
        ax_twin.legend(fontsize='large')

# Create a figure
fig = plt.figure(figsize=(24, 24))

# Define the GridSpec layout
gs = GridSpec(6, 4, figure=fig)

# order = [[42, 'twin', 48],
#          [43, 'twin', 49],
#          [44, 'twin', 50],
#          [45, 'twin', 51],
#          [46, 'twin', 52],
#          [47, 'twin', 53],
#          ['violin'],
#         ]

order = [[0, 'twin', 6],
         [1, 'twin', 7],
         [2, 'twin', 8],
         [3, 'twin', 9],
         [4, 'twin', 10],
         [5, 'twin', 11],
         ['violin'],
        ]


# List of axes indices in GridSpec for each subplot
subplot_specs = [(0, 1, 0, 1), # a
                 (0, 1, 1, 2), # b
                 (1, 2, 0, 1), # c
                 (1, 2, 1, 2), # d
                 (2, 3, 0, 1), # e
                 (2, 3, 1, 2), # f
                 (0, 3, 2, 6), # g
                ]

# Create and set up subplots
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    ax = fig.add_subplot(gs[r_start:r_end, c_start:c_end])
    if i < len(axes):
        idx = order[i]
        if idx[0] == 'violin':
            df = pd.DataFrame()

            # scales the parameters
            scaled_param = dataset.SHO_scaler.transform(parm)

            # gets the parameters from the SHO LSQF fit
            true = dataset.SHO_fit_results().reshape(-1, 4)

            # Builds the dataframe for the violin plot
            true_df = pd.DataFrame(
                true, columns=["Amplitude", "Resonance", "Q-Factor", "Phase"]
            )
            predicted_df = pd.DataFrame(
                scaled_param, columns=["Amplitude",
                                       "Resonance", "Q-Factor", "Phase"]
            )

            # merges the two dataframes
            df = pd.concat((true_df, predicted_df))

            # adds the labels to the dataframe
            names = [true, scaled_param]
            names_str = ["LSQF", "NN"]
            labels = ["A", "\u03C9", "Q", "\u03C6"]

            # adds the labels to the dataframe
            for j, name in enumerate(names):
                for i, label in enumerate(labels):
                    dict_ = {
                        "value": name[:, i],
                        "parameter": np.repeat(label, name.shape[0]),
                        "dataset": np.repeat(names_str[j], name.shape[0]),
                    }

                    df = pd.concat((df, pd.DataFrame(dict_)))
              
              # Reset index to handle potential duplicated columns or indices
            df = df.reset_index(drop=False)
            
            # plots the data
            sns.violinplot(
                data=df, x="parameter", y="value", hue="dataset", split=True, ax=ax
            )

            # labels the figure and does some styling
            labelfigs(ax, 0, style="b")
            ax.set_ylabel("Scaled SHO Results")
            ax.set_xlabel("")

            # Get the legend associated with the plot
            legend = ax.get_legend()
            legend.set_title("")
            plt.setp(legend.get_texts(), fontsize='large') # Set the label size
        else:
            copy_axes_properties(axes[idx[0]], ax, axes[idx[2]], idx[1])
             #copy_axes_properties(axes[r_start], ax, axes[r_end], idx[1])


# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()

fig = BE_viz.SHO_switching_maps(parm);

# Set the figure size with 24 inches width
fig.set_size_inches(24, 10, forward=True)

# Show the layout
fig.show()

In [ ]:
len(dataset.dc_voltage)

In [ ]:
len(dataset.get_cycle(dataset.dc_voltage))

In [ ]:
dataset.num_cycles

In [ ]:
dataset.get_voltage.shape

In [ ]:
dataset.cycle = 2

In [ ]:
plt.plot(dataset.get_voltage,color = 'blue')
plt.plot(dataset.dc_voltage,color = 'orange')
plt.plot(dataset.get_cycle(dataset.dc_voltage),color = 'black')

In [ ]:
plt

## This might be part of the problem with the voltages. `num_cycles == 4` and `dc_voltage` has 192 steps so it results in only 48 total steps, not the expected 96

In [ ]:
BE_viz.SHO_switching_maps(parm,
                          clims=[
                                    (0, 1.4e-4),  # amplitude
                                    (1.31e6, 1.33e6),  # resonance frequency
                                    (-230, -160),  # quality factor
                                    (0, 2*np.pi),  # phase
                                ], 
                          cycle=3,
                          
                          
                          
                          )

In [ ]:
BE_viz.SHO_switching_maps(parm,
                          clims=[
                                    (0, 1.4e-4),  # amplitude
                                    (1.31e6, 1.33e6),  # resonance frequency
                                    (-230, -160),  # quality factor
                                    (0, 2*np.pi),  # phase
                                ], 
                          cycle=3,
                          
                          
                          
                          );

Go from source code. Run the SHO_fit_results

Figure out how to do it for the voltage points in the figure. Use the `state` keyword?

state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}

LSQF_ = {'resampled': True,
                'raw_format': 'complex',
                'fitter': 'LSQF',
                'scaled': True,
                'output_shape': 'index',
                'measurement_state': 'all',
                'resampled_bins': 165,
                'LSQF_phase_shift': 1.5707963267948966,
                'NN_phase_shift': None,
                'noise': noise}


In [ ]:
true_state

In [ ]:
out_state

In [ ]:
test = dataset.SHO_fit_results(state = out_state, model = model)

In [ ]:
test.shape

In [ ]:
test[:,0].reshape(60,60,96,-1).shape

In [ ]:
plt.imshow(test[:,0].reshape(60,60,96,-1)[:,:,0,0])

In [ ]:
idx[0]

In [ ]:
len(axes)

Figure 3. SHO fitting results of DNN in comparison with LSQF method’s results. a,c,e Best, median and worst predictions of LSQF method. b,d,f Best, median and worst predictions of neural network trained with ADAHESSIAN. g Distributions of predicted parameters. h Band-excitation waveform with switching voltage comparing maps of four parameters (i-q) predicted by NN and LSQF

## Figure 4

In [ ]:
h5_loop_fit, h5_loop_group = dataset.LSQF_Loop_Fit()

In [ ]:
%load_ext autoreload
%autoreload 2

from belearn.dataset.dataset import BE_Dataset
from belearn.viz.viz import Viz
from m3util.viz.printing import printer
printing = printer(basepath = './Figures/')


In [ ]:
# instantiate the visualization object
image_scalebar = [2000, 500, "nm", "br"]


# Specify the filename and the path to save the file
filename = "data_raw.h5"
save_path = "./Data"

data_path = save_path + "/" + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path)


In [ ]:

BE_viz = Viz(dataset, printing, verbose=True, 
             SHO_ranges = [(0,1.5e-4), (1.31e6, 1.33e6), (-300, 300), (-np.pi, np.pi)], 
             image_scalebar = image_scalebar)

In [ ]:

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

In [ ]:
#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim

datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(BE_viz.loop_fitting_function_torch, # function 
                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/jca92/Rapid-Fitting-BEPFM-NN/notebooks/7_Hysteresis_Fitter.ipynb")

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": "cuda",
    "ADAM_epochs": 100}


train =  True

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 500,
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.00602861393450035.pth"
    )

In [ ]:
from m3util.viz.style import set_style

set_style("printing")


In [ ]:
n = 1

data = ("LSQF", "NN")

fig = BE_viz.hysteresis_comparison(
    data,
    nn_model=model,
    filename="Figure_XX_LSQF_NN_bmw_comparison",
)

In [ ]:
fig

In [ ]:
axes = fig.axes

In [ ]:
axes

In [ ]:
import torch

In [ ]:
data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)

fig = BE_viz.violin_plot_comparison_hysteresis(model,
                                         torch.atleast_3d(torch.tensor(data.reshape(-1, 96))),
                                         filename="Figure_XX_Violin") 

In [ ]:
fig.axes

In [ ]:
axes.extend(fig.axes)

In [ ]:
data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
data = torch.atleast_3d(torch.tensor(data.reshape(-1, 96)))

pred_recon, pred_params_scaled, pred_params = model.predict(
    data,
    1024,
    translate_params=False,
    is_SHO=False
)

fig = BE_viz.hysteresis_maps(pred_params, cycle=0, filename="Figure_XX_NN_Hysteresis_Maps")

In [ ]:
plt.rcParams["xtick.labelsize"]

In [ ]:
axes

In [ ]:
for line0 in axes[0].get_lines():
    x_data_0 = line0.get_xdata()
    y_data_0 = line0.get_ydata()

for line1 in axes[1].get_lines():
    x_data_1 = line1.get_xdata()
    y_data_1 = line1.get_ydata()

print(min(x_data_0==x_data_1),min(y_data_0==y_data_1) )



In [ ]:
y_data_0

In [ ]:
y_data_1

In [ ]:
for line in axes[3].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        
        plt.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
plt.show()

In [ ]:
np.linspace(np.min(line.get_xdata()),np.max(line.get_xdata()),9)

In [ ]:
[0,1,2,3,4,5,6][-7]

In [ ]:
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    print(i,-i-2)

In [ ]:
ax.get_h

In [ ]:
import pandas as pd
import seaborn as sns
import itertools
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front



def copy_axes_properties(source_ax, target_ax, secondary_ax=None, type_ax=None):
    """Copy properties and data from source_ax to target_ax."""
    # Copy basic properties
    target_ax.set_xlabel(source_ax.get_xlabel(),fontsize=14)
    target_ax.set_ylabel(source_ax.get_ylabel(),fontsize=14)
    target_ax.set_xlim(source_ax.get_xlim())
    target_ax.set_ylim(source_ax.get_ylim())
    target_ax.set_title(source_ax.get_title())

    # Copy lines from the primary axis
    for line in source_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Handle twin axes if present
    if secondary_ax:
        ax_twin = target_ax.twinx()
        ax_twin.set_ylim(secondary_ax.get_ylim())
        ax_twin.set_ylabel(secondary_ax.get_ylabel(),fontsize=14)

        for line in secondary_ax.get_lines():
            label = line.get_label() if line.get_label() != '_nolegend_' else None
            # Copying line properties for the twin axis
            ax_twin.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                         linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Copy legends
    if source_ax.get_legend():
        target_ax.legend(fontsize='large')

    if secondary_ax and secondary_ax.get_legend():
        ax_twin.legend(fontsize='large')
        
    #BE_viz._scientific_notation_dual(target_ax,ax_twin)
    
    
    
    set_sci_notation_label(
                target_ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5,
                textsize = 8,
            )
    
    # set_sci_notation_label(
    #             ax_twin, corner="top right", axis="y", stroke_color="w", linewidth=0.5,
    #             textsize=8,
    #         )
    
    target_ax.tick_params(axis='x',labelsize=10)
    target_ax.tick_params(axis='y',labelsize=10)

    # ax_twin.tick_params(axis='x',labelsize=10)
    # ax_twin.tick_params(axis='y',labelsize=10)

    target_ax.set_xticks(np.linspace(np.min(line.get_xdata()),np.max(line.get_xdata()),9))

# Create a figure
fig = plt.figure(figsize=(16, 16))

# Define the GridSpec layout
gs = GridSpec(6, 4, figure=fig)

order = [[0, 'twin', 6],
         [1, 'twin', 7],
         [2, 'twin', 8],
         [3, 'twin', 9],
         [4, 'twin', 10],
         [5, 'twin', 11],
         ['violin'],
        ]

# List of axes indices in GridSpec for each subplot
subplot_specs = [(0, 1, 0, 1), # a
                 (0, 1, 1, 2), # b
                 (1, 2, 0, 1), # c
                 (1, 2, 1, 2), # d
                 (2, 3, 0, 1), # e
                 (2, 3, 1, 2), # f
                 (0, 3, 2, 6), # g
                ]

# Create and set up subplots
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    ax = fig.add_subplot(gs[r_start:r_end, c_start:c_end])
    if i < len(axes):
        idx = order[i]
        if idx[0] == 'violin':
            df = pd.DataFrame()

            # uses the model to get the predictions
            pred_data, scaled_param, params = model.predict(data, is_SHO=False)

            true = dataset.LSQF_hysteresis_params().reshape(-1, 9)

            true_scaled = dataset.loop_param_scaler.transform(true)

            # Builds the dataframe for the violin plot
            true_df = pd.DataFrame(
                true, columns=["a0", "a1", "a2", "a3", "a4",
                               "b0", "b1", "b2", "b3"]
            )
            predicted_df = pd.DataFrame(
                scaled_param, columns=["a0", "a1", "a2", "a3", "a4",
                                       "b0", "b1", "b2", "b3"]
            )

            # merges the two dataframes
            df = pd.concat((predicted_df, true_df))

            # adds the labels to the dataframe
            names = [true_scaled, scaled_param]
            names_str = ["NN", "LSQF"]
            labels = ["a0", "a1", "a2", "a3", "a4", "b0", "b1", "b2", "b3"]

            # adds the labels to the dataframe
            for j, name in enumerate(names):
                for i, label in enumerate(labels):
                    dict_ = {
                        "value": name[:, i],
                        "parameter": np.repeat(label, name.shape[0]),
                        "dataset": np.repeat(names_str[j], name.shape[0]),
                    }

                    df = pd.concat((df, pd.DataFrame(dict_)))
                    
            
                    

#             # builds the plot
#             fig, ax = plt.subplots(figsize=(4, 4))

              # Reset index to handle potential duplicated columns or indices
            df = df.reset_index(drop=False)
            
            # plots the data
            sns.violinplot(
                data=df, x="parameter", y="value", hue="dataset", split=True, ax=ax
            )

            # labels the figure and does some styling
            labelfigs(ax, string_add = 'g', loc ='tl',size=20, style="b", inset_fraction=(0.05,0.12))
            ax.set_ylabel("Scaled Hysteresis Results",fontsize=16)
            ax.set_xlabel("")
            
            ax.tick_params(axis='x',labelsize=14)
            ax.tick_params(axis='y',labelsize=14)

            # Get the legend associated with the plot
            legend = ax.get_legend()
            legend.set_title("")
            plt.setp(legend.get_texts(), fontsize='large') # Set the label size
        else:
           # copy_axes_properties(axes[idx[0]], ax, axes[idx[2]], idx[1])
            #copy_axes_properties(axes[r_start], ax, axes[r_end], idx[1])
            #copy_axes_properties(axes[r_start], ax)
            for line in axes[-i-2].get_lines():
                    label = line.get_label() if line.get_label() != '_nolegend_' else None
                    # Copying line properties like color, linestyle, marker, etc.
                    
                    ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                                linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
                    
                   # ax.legend(fontsize='large')
            

            set_sci_notation_label(
                ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5,
                textsize = 8,
            )
            
            
            ax.tick_params(axis='x',labelsize=10)
            ax.tick_params(axis='y',labelsize=10)

            # ax_twin.tick_params(axis='x',labelsize=10)
            # ax_twin.tick_params(axis='y',labelsize=10)

            ax.set_xticks(np.linspace(np.min(line.get_xdata()),np.max(line.get_xdata()),9))
            
            if i in [0,1]:
              
                labelfigs(ax,
                        string_add="Best",
                        loc ='tl',
                        size=8,
                        inset_fraction=(0.04,0.5),
                        style = 'b'
                        )
                
               # get legend handles and their corresponding labels
                handles, labels = ax.get_legend_handles_labels()

                # zip labels as keys and handles as values into a dictionary, ...
                # so only unique labels would be stored 
                dict_of_labels = dict(zip(labels, handles))

                # use unique labels (dict_of_labels.keys()) to generate your legend
                ax.legend(dict_of_labels.values(), dict_of_labels.keys(),fontsize='large',
                          loc = (0.02,0.65))#, bbox_to_anchor = (0,0.50))

            elif i in [2,3]:
                labelfigs(ax,
                        string_add="Median",
                        loc ='tl',
                        size=8,
                        inset_fraction=(0.04,0.5),
                        style = 'b'
                        )
                
            elif i in [4,5]:
                labelfigs(ax,
                        string_add="Worst",
                        loc ='tl',
                        size=8,
                        inset_fraction=(0.04,0.5),
                        style = 'b'
                        )
  
            
            labelfigs(ax,
                number=i,
                loc ='tl',
                size=12,
                inset_fraction=(0.05,0.15),
                style = 'b'
                )

            # if i == 0:
            #     ax.set_title("Least Square Fit")
            # elif i==1:
            #     ax.set_title("NN with Trust Region CG")
            
            # set_sci_notation_label(
            #     ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5
            # )
            
            # set_sci_notation_label(
            #     ax, corner="top right", axis="y", stroke_color="w", linewidth=0.5
            # )
            

            

# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()


filename = "Figure_4a"

BE_viz.Printer.savefig(
                fig, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        


data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
data = torch.atleast_3d(torch.tensor(data.reshape(-1, 96)))

pred_recon, pred_params_scaled, pred_params = model.predict(
    data,
    1024,
    translate_params=False,
    is_SHO=False
)
fig = BE_viz.hysteresis_maps(pred_params, cycle=0);

# Set the figure size with 24 inches width
fig.set_size_inches(24, 10, forward=True)


filename = "Figure_4b"


# for i in range(len(fig.axes)):
#     if fig.axes[i].get_xlabel() == "a0":
#         label_index = i
        


labelfigs(fig.axes[0],
        string_add="h",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )


labelfigs(fig.axes[9],
        string_add="i",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )

BE_viz.Printer.savefig(
                fig, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        



# Show the layout
fig.show()

In [ ]:
fig = BE_viz.hysteresis_maps(pred_params, cycle=0);

# Set the figure size with 24 inches width
fig.set_size_inches(24, 10, forward=True)


filename = "Figure_4b"


# for i in range(len(fig.axes)):
#     if fig.axes[i].get_xlabel() == "a0":
#         label_index = i
        


labelfigs(fig.axes[0],
        string_add="h",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )


labelfigs(fig.axes[9],
        string_add="i",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )

BE_viz.Printer.savefig(
                fig, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        



# Show the layout
fig.show()

In [ ]:
for i in range(len(fig.axes)):
    if fig.axes[i].get_xlabel() == "a0":
        print(i)

In [ ]:
fig.axes[20].get_xlabel()

In [ ]:
"Axes" in np.array(fig.axes[0])

In [ ]:
np.where("a0" in fig.axes)

In [ ]:
fig.axes[np.where("a0" in fig.axes)[0]]

In [ ]:
import pandas as pd
import seaborn as sns
import itertools
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front



def copy_axes_properties(source_ax, target_ax, secondary_ax=None, type_ax=None):
    """Copy properties and data from source_ax to target_ax."""
    # Copy basic properties
    target_ax.set_xlabel(source_ax.get_xlabel())
    target_ax.set_ylabel(source_ax.get_ylabel())
    target_ax.set_xlim(source_ax.get_xlim())
    target_ax.set_ylim(source_ax.get_ylim())
    target_ax.set_title(source_ax.get_title())

    # Copy lines from the primary axis
    for line in source_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Handle twin axes if present
    if secondary_ax:
        ax_twin = target_ax.twinx()
        ax_twin.set_ylim(secondary_ax.get_ylim())
        ax_twin.set_ylabel(secondary_ax.get_ylabel())

        for line in secondary_ax.get_lines():
            label = line.get_label() if line.get_label() != '_nolegend_' else None
            # Copying line properties for the twin axis
            ax_twin.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                         linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Copy legends
    if source_ax.get_legend():
        target_ax.legend(fontsize='large')

    if secondary_ax and secondary_ax.get_legend():
        ax_twin.legend(fontsize='large')

# Create a figure
fig2 = plt.figure(figsize=(24, 24))

# Define the GridSpec layout
gs2 = GridSpec(10, 4, figure=fig2)

order2 = [[0, 'twin', 6],
         [1, 'twin', 7],
         [2, 'twin', 8],
         [3, 'twin', 9],
         [4, 'twin', 10],
         [5, 'twin', 11],
         ['violin'],
         ['hysteresis_maps']
        ]

# List of axes indices in GridSpec for each subplot
subplot_specs2 = [(0, 1, 0, 1), # a
                 (0, 1, 1, 2), # b
                 (1, 2, 0, 1), # c
                 (1, 2, 1, 2), # d
                 (2, 3, 0, 1), # e
                 (2, 3, 1, 2), # f
                 (0, 3, 2, 6),
                 (3, 6, 0, 5), # h & i together
                ]

# Create and set up subplots
for i2, (r_start2, r_end2, c_start2, c_end2) in enumerate(subplot_specs2):
        ax2 = fig2.add_subplot(gs2[r_start2:r_end2, c_start2:c_end2])
   # if i2 < len(axes):
        idx2 = order2[i2]
        if idx2[0] == 'violin':
            df = pd.DataFrame()

            # uses the model to get the predictions
            pred_data, scaled_param, params = model.predict(data, is_SHO=False)

            true = dataset.LSQF_hysteresis_params().reshape(-1, 9)

            true_scaled = dataset.loop_param_scaler.transform(true)

            # Builds the dataframe for the violin plot
            true_df = pd.DataFrame(
                true, columns=["a0", "a1", "a2", "a3", "a4",
                               "b0", "b1", "b2", "b3"]
            )
            predicted_df = pd.DataFrame(
                scaled_param, columns=["a0", "a1", "a2", "a3", "a4",
                                       "b0", "b1", "b2", "b3"]
            )

            # merges the two dataframes
            df = pd.concat((predicted_df, true_df))

            # adds the labels to the dataframe
            names = [true_scaled, scaled_param]
            names_str = ["NN", "LSQF"]
            labels = ["a0", "a1", "a2", "a3", "a4", "b0", "b1", "b2", "b3"]

            # adds the labels to the dataframe
            for j, name in enumerate(names):
                for i, label in enumerate(labels):
                    dict_ = {
                        "value": name[:, i],
                        "parameter": np.repeat(label, name.shape[0]),
                        "dataset": np.repeat(names_str[j], name.shape[0]),
                    }

                    df = pd.concat((df, pd.DataFrame(dict_)))
                    
            
                    

#             # builds the plot
#             fig, ax = plt.subplots(figsize=(4, 4))

              # Reset index to handle potential duplicated columns or indices
            df = df.reset_index(drop=False)
            
            # plots the data
            sns.violinplot(
                data=df, x="parameter", y="value", hue="dataset", split=True, ax=ax2
            )

            # labels the figure and does some styling
            labelfigs(ax2, 0, style="b")
            ax2.set_ylabel("Scaled Hysteresis Results")
            ax2.set_xlabel("")

            # Get the legend associated with the plot
            legend2 = ax2.get_legend()
            legend2.set_title("")
            plt.setp(legend2.get_texts(), fontsize='large') # Set the label size
            
            
        elif idx2[0] == "hysteresis_maps":
            data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
            data = torch.atleast_3d(torch.tensor(data.reshape(-1, 96)))

            pred_recon2, pred_params_scaled2, pred_params2 = model.predict(
                data,
                1024,
                translate_params=False,
                is_SHO=False
            )
            
            # something about figure 
            ax2.axes=BE_viz.hysteresis_maps(pred_params, cycle=0)[1]
  
            
        else:
           # copy_axes_properties(axes[idx[0]], ax, axes[idx[2]], idx[1])
           copy_axes_properties(axes[r_start2], ax2, axes[r_end2], idx2[1])
            

# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()



#fig = BE_viz.hysteresis_maps(pred_params, cycle=0);

# Set the figure size with 24 inches width
fig2.set_size_inches(24, 10, forward=True)


filename = "Figure_4_TEST"

BE_viz.Printer.savefig(
                fig2, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        



# Show the layout
fig2.show()

In [ ]:
ax2.axis()

In [ ]:
fig2.

In [ ]:
BE_viz.hysteresis_maps(pred_params, cycle=0)[1].

In [ ]:
list(BE_viz.hysteresis_maps(pred_params, cycle=0)[1])

In [ ]:
ax2.axis = BE_viz.hysteresis_maps(pred_params, cycle=0)[1]

In [ ]:
ax2.set_figure

In [ ]:
ax2.axes.figure.axes.append(BE_viz.hysteresis_maps(pred_params, cycle=0)[1])

In [ ]:
type(BE_viz.hysteresis_maps(pred_params, cycle=0).axes[5])

In [ ]:
for i2, (r_start2, r_end2, c_start2, c_end2) in enumerate(subplot_specs2):
    print(i2,order2[i2],r_start2,r_end2,c_start2,c_end2)

In [ ]:
# Create a figure
fig2 = plt.figure(figsize=(24, 24))

# Define the GridSpec layout
gs2 = GridSpec(6, 4, figure=fig)


In [ ]:
df["parameter"]

In [ ]:
df["value"]

In [ ]:
df["value"]
0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
            ...   
287995   -0.601739
287996   -0.685297
287997   -0.603866
287998   -0.524238
287999   -0.646338
Name: value, Length: 288000, dtype: float64

In [ ]:
dataset.LSQF_hysteresis_params().shape
(60, 60, 4, 9)

In [ ]:
axes[idx[0]]

In [ ]:
len(axes)

Figure 4. Piezoelectric hysteresis loops fitting results of DNN in comparison with LSQF method’s results. a,c,e Best, median, and worst predictions of LSQF method. b,d,f Best, median, and worst predictions of neural network trained with Trust Region CG. g Distributions of predicted parameters. h Color maps of the signal of parameters resulted from LSQF method. i Color maps of the signal of parameters resulted from neural network.

## Figure 5

In [ ]:
fig, axs = plt.subplots(
            1,
            3,
            figsize=(12, 4),
            #gridspec_kw={"height_ratios": [1, 1]},
        )

